Importaciones de librerías necesarias para el código

In [1]:
import torch
from ultralytics import YOLO
import cv2
import csv

# Detector de matrículas

In [4]:
VIDEO_PATH = "C0142.MP4"
OUTPUT_PATH = "salida.mp4"
OUT_WIDTH = 1280          # El alto se calcula para mantener la proporción
IMGSZ = 640               # Tamaño de procesamiento del modelo (960 para objetos pequeños)
SHOW = True               # Pon False si no tienes entorno gráfico (Colab, servidor)
SAVE = True               # Guardar el vídeo resultante

# COCO: 0=person, 2=car, 3=motorcycle, 5=bus
CLASSES = [0, 2, 3, 5]
VEHICLES = {"car", "bus", "motorcycle"}

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

OUT_HEIGHT = int(src_h * OUT_WIDTH / src_w)

writer = None
if SAVE:
    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, (OUT_WIDTH, OUT_HEIGHT))

if SHOW:
    cv2.namedWindow("Tracking", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Tracking", OUT_WIDTH, OUT_HEIGHT)

person_ids = set()
vehicle_ids = set()
frame_idx = 0

results = model.track(source=VIDEO_PATH, stream=True, persist=True,
                      classes=CLASSES, imgsz=IMGSZ, show=False, verbose=False)

csv_file = open("tracking.csv", "w", newline="", encoding="utf-8")
writercsv = csv.writer(csv_file)
writercsv.writerow(["fotograma", "tipo_objeto", "confianza", "identificador_tracking", "x1", "y1", "x2", "y2",
                    "matrícula_en_su_caso", "confianza", "mx1", "my1", "mx2", "my2", "texto_matricula"])      # cabecera

for r in results:
    # Frame con cajas e IDs, redimensionado
    frame = r.plot()
    frame = cv2.resize(frame, (OUT_WIDTH, OUT_HEIGHT))
    h, w = frame.shape[:2]

    # Contar IDs únicos
    for box in r.boxes:
        if box.id is None:
            continue

        name = model.names[int(box.cls[0])]
        track_id = int(box.id[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        plate = ""
        trust = float(box.conf[0])
        mplate = ""
        mtrust = ""
        mx1, my1, mx2, my2 = "", "", "", ""
        text_plate = ""

        if name in VEHICLES:
            vehicle_ids.add(track_id)

            x1, y1 = max(x1, 0), max(y1, 0)
            x2, y2 = min(x2, w), min(y2, h)
            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            # IMPORTANTE: predict, no track (los recortes cambian de tamaño y no son un vídeo continuo)
            # rb = second_model.predict(crop, verbose=False)[0]
        elif name == "person":
            person_ids.add(track_id)

        writercsv.writerow([
            frame_idx,
            name,
            trust,
            track_id,
            x1, y1, x2, y2,
            plate,
            mtrust,
            mx1, my1, mx2, my2,
            text_plate
        ])

    # Contadores (se dibujan tras redimensionar para que el texto no se deforme)
    cv2.rectangle(frame, (10, 10), (280, 90), (0, 0, 0), -1)
    cv2.putText(frame, f"Vehiculos: {len(vehicle_ids)}", (20, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(frame, f"Personas: {len(person_ids)}", (20, 78),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    if writer:
        writer.write(frame)

    if SHOW:
        cv2.imshow("Tracking", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    frame_idx += 1

csv_file.close()

if writer:
    writer.release()
if SHOW:
    cv2.destroyAllWindows()

KeyboardInterrupt: 

In [ ]:
model = YOLO("runs/detect/train-2/weights/best.pt")
results = model.track(source="C0142.MP4", show=True)

# Implementación de OCR sobre las matrículas detectadas